# DHN GHL — Math ↔ RelNN DSL

This notebook maps the **Deep Homomorphism Networks (DHN)** equations from the paper to the corresponding RelNN implementations.

## Canonical configuration: C2:4 (Edge-join)

**Program:** `dhn_ghl_csl_c2_4.relnn` (inline edge-join enumeration, elegant and paper-faithful)

**Data:** `build_dhn_db_edge(graphs, ...) → Node, Edge`

Homomorphism instances are computed inline via joins over the `Edge` relation, matching the paper's mathematical definition without preprocessing.

## C2:10 (Simple-cycles precompute)

For larger pattern sets (C2 through C10), memory constraints require precomputing homomorphisms.

**Program:** `dhn_ghl_csl_c2_10.relnn` (templated with C2..C10)

**Data:** `build_dhn_db(..., patterns=["C2",...,"C10"], simple_cycles=True) → Node, Hom_C2, ..., Hom_C10`

Uses `nx.simple_cycles` for enumeration (matches official gear/dhn strategy for tractability).

---

## Layer 1: Homomorphism enumeration + transformation

**Paper Equation 3:** For a pattern $P$ (e.g., $C_k$), root node $u$, and homomorphism $h: P \to G$:

$$m_u^{(k)} = \sum_{h: P \to G, h(0)=u} \prod_{i=0}^{|P|-1} \mu_i(h_v^{(0)}) \cdot \mathbb{1}[h \text{ valid}]$$

where $\mu_i$ is a per-position MLP and $h_v^{(0)}$ is the node embedding at position $i$ of the homomorphism.

### In RelNN (C2:4 edge-join):

```relnn
# Position-wise transforms: multiply at root by homomorphism indicator
C2_T0(graph_id, u, v; Mu<'C2', 0>(...) * w) :- Hom_C2(graph_id, u, v; w), H0(graph_id, u; z) .
C2_T1(graph_id, u, v; Mu<'C2', 1>(...))     :- Hom_C2(graph_id, u, v; _), H0(graph_id, v; z) .

# Product + aggregation
C2_Agg(graph_id, u; sum(z0 * z1)) :- C2_T0(...; z0), C2_T1(...; z1) .
```

where `Hom_C2` is computed via the join:

```relnn
# Inline: Edge(u,v) represents C2 (u → v homomorphism)
# The join naturally enumerates all rooted paths in the graph
```

**Key:** `Hom_C2(graph_id, u, v; w)` has embedding $w \in \{0, 1\}$ (real homomorphism or dummy padding for nodes without matches).

## Layer 2: Pattern combination and graph readout

**Paper Equation 4:** Combine pattern embeddings per node via a learnable transformation $\rho$:

$$h_u^{(1)} = \rho\left(\text{Concat}\left(m_u^{(2)}, m_u^{(3)}, m_u^{(4)}\right)\right)$$

where $\rho$ is a 3-layer MLP with ReLU activations.

### In RelNN:

```relnn
H1(graph_id, n; Rho_L3(ReLU()(Rho_L2(ReLU()(Rho_L1(
  Concat(z0, z1, z2)
)))))) :- C2_Agg(graph_id, n; z0), C3_Agg(graph_id, n; z1), C4_Agg(graph_id, n; z2) .
```

**Graph-level readout** (pooling over nodes):

$$e_g = \sum_n h_n^{(1)}$$

```relnn
GraphEmb(graph_id; sum(z)) :- H1(graph_id, n; z) .
GraphLogits(graph_id; Classifier(z)) :- GraphEmb(graph_id; z) .
```

## Homomorphism representation

### Edge-join (C2:4)

In the canonical C2:4 setup, `Hom_C*` relations are derived inline from joins:

| Pattern | Homomorphism source | Embedding |
|---------|---------------------|----------|
| C2 | `Edge(u,v)` → `(u,v)` tuple | $w \in \{0, 1\}$ (1 for real edges, 0 for padding) |
| C3 | Join `Edge(u,v), Edge(v,w)` → `(u,v,w)` | 1 for valid cycles, 0 for padding |
| C4 | Join `Edge(u,v), Edge(v,w), Edge(w,x)` → `(u,v,w,x)` | 1 for valid cycles, 0 for padding |

### Simple-cycles precompute (C2:10)

For C2:10, `Hom_C*` tables are precomputed using `nx.simple_cycles` (matching official gear/dhn):

```python
db = build_dhn_db(
    graphs,
    ["C2", "C3", ..., "C10"],
    simple_cycles=True  # Use nx.simple_cycles + rotations
)
```

This materializes all rooted simple cycles up to length 10, significantly reducing memory vs. exhaustive homomorphism enumeration.

## MLP templates

**Per-position MLP** $\mu_{k,i}$:

$$\mu_{k,i}(z) = W^{(3)}_{k,i} \,\mathrm{ReLU}\big(W^{(2)}_{k,i} \,\mathrm{ReLU}(W^{(1)}_{k,i} z)\big)$$

In RelNN, templated as `Mu_L1/L2/L3<k, i>` (Linear layers stacked with ReLU):

```relnn
Mu_L1<k, i> = Linear(d_in, d_hidden) .
Mu_L2<k, i> = Linear(d_hidden, d_hidden) .
Mu_L3<k, i> = Linear(d_hidden, d_k) .
```

**Pattern combination (Rho):**

$$\rho(u) = W^{(3)}_\rho \,\mathrm{ReLU}(W^{(2)}_\rho \,\mathrm{ReLU}(W^{(1)}_\rho u))$$

```relnn
Rho_L1 = Linear(input_dim, d_hidden) .
Rho_L2 = Linear(d_hidden, d_hidden) .
Rho_L3 = Linear(d_hidden, d_k) .
```

where `input_dim = num_patterns * d_k` (e.g., 30 for C2:4 with 3 patterns × 10 dims).

In [ ]:
# Open the canonical RelNN program
from pathlib import Path
prog_c2_4 = Path("dhn_ghl_csl_c2_4.relnn").read_text(encoding="utf-8")
print(f"Lines: {len(prog_c2_4.splitlines())}")
print("\n--- First 50 lines ---\n")
for i, line in enumerate(prog_c2_4.splitlines()[:50], 1):
    print(f"{i:3d}  {line}")